[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [asyncpg and psycopg3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)

# Types and Adaptation


## What you will be able to do

Say what happens to a value between your variable and the column, in both directions, and why every
one of them becomes bytes on the way. Name the Python types that already have a rule and recognize
the ones that do not. Write a rule for a class of your own, in both directions, and register it.
Know why a `numeric` column comes back as `Decimal` and what breaks when that meets a float, why a
timestamp moves when the session time zone changes, and why a large enough integer stops being an
integer. And do the same in asyncpg, where a JSONB column arrives as a string until you say
otherwise.


## The idea

### The problem

The socket carries bytes. Nothing else crosses it, in either direction, so every value you pass and
every value you get back has been through a conversion that somebody wrote.

For most types somebody already did: `int`, `str`, `datetime`, `list` and a dozen more have rules in
both drivers. For a class of your own there is no rule, and psycopg says so rather than guessing:
`cannot adapt type 'Money'`. That is the good case. The harder cases are the types that do have a
rule, and whose rule is not what you assumed.

### What adaptation is

Two directions with two names. A **dumper** turns a Python object into the bytes for a PostgreSQL
type on the way in. A **loader** turns the bytes of a PostgreSQL type into a Python object on the
way out. psycopg keeps a registry of both, which you can add to per connection or globally.

asyncpg calls the pair a **codec**, registers both halves at once with `set_type_codec`, and ships
with fewer of them, which is why one of its defaults surprises people: a JSONB column comes back as
the string it was stored as.

### Why it works that way

A driver cannot guess what a class means. `Money("19.99")` might belong in a `numeric`, or in a
`text`, or in two columns, and getting it wrong silently would be worse than refusing. So psycopg
refuses, and asking you to write four lines is the whole of the fix.

The surprises in the other direction come from PostgreSQL being stricter than Python about numbers.
`numeric` is exact, so it loads as `Decimal` rather than `float`, and a `Decimal` will not mix with a
`float` without being told to.

### Where this shows up

Money, timestamps and anything with a class around it. The `Decimal` one shows up as a `TypeError`
in a calculation nowhere near the query, and the time zone one shows up as a report that is right
locally and wrong in production.

### What this notebook covers

What already has a rule. The three that catch people: `numeric` as `Decimal`, `timestamptz` and the
session time zone, and an integer too large to be one. Writing a dumper and a loader for a class of
your own. Arrays, which are what made `= ANY` work in **Placeholders and Identifiers**. Then
asyncpg's codecs and the JSONB default. Then the four failures.

Enums have a registration mechanism of their own, `register_enum`, layered on top of the dumper this
notebook teaches. It is named here and not taught.

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import psycopg
from psycopg.adapt import Dumper


class Money:
    def __init__(self, amount):
        self.amount = amount


class MoneyDumper(Dumper):
    oid = psycopg.adapters.types["numeric"].oid         # what the server should call it

    def dump(self, obj):
        return str(obj.amount).encode()                 # every value becomes bytes


with psycopg.connect("dbname=guide") as conn:
    try:
        conn.execute("SELECT %s", (Money("19.99"),))
    except psycopg.ProgrammingError as error:
        print("with no rule:", error)
    conn.rollback()

    conn.adapters.register_dumper(Money, MoneyDumper)
    print("with a rule: ", conn.execute("SELECT %s", (Money("19.99"),)).fetchone())
```

```
with no rule: cannot adapt type 'Money' using placeholder '%s' (format: AUTO)
with a rule:  (Decimal('19.99'),)
```

Four lines of rule, and the class crosses the socket. Note what came back: a `Decimal`, not a
`Money`, because a dumper only describes the way in. The way back needs a loader, which is further
down.


## Setup

Twelve imports, both drivers, the server, and a table with one row of interesting types.

- `psycopg` and `asyncpg` are the drivers, and `Dumper` and `Loader`, from `psycopg.adapt`, are the
  two halves of a rule
- `Decimal`, `datetime` and `json` are the Python types this notebook meets
- `subprocess`, `sys`, `os`, `getpass` and `time` stand the server up, which **A Server of Your Own**
  takes apart
- `version` and `PackageNotFoundError` install the drivers where they are missing

`Money` is the class with no rule. `prices` holds one row with a `numeric`, a `timestamptz` and a
`text[]`, which are the three that surprise people. `utc` opens a connection whose session time zone
is UTC, which every cell printing a timestamp below uses, because the default comes from the machine
and would otherwise print something different for you than it does here.


In [1]:
import datetime
import getpass
import json
import os
import subprocess
import sys
import time
from decimal import Decimal
from importlib.metadata import PackageNotFoundError, version

try:
    if version("psycopg") < "3.3" or version("asyncpg") < "0.31":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "psycopg[binary,pool]==3.3.6", "psycopg-pool==3.3.2", "asyncpg==0.31.0"],
                   check=True)

import asyncpg
import psycopg
from psycopg.adapt import Dumper, Loader

def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(database="postgres"):
    """Whether a server is there, asked the only way that needs no client binaries."""
    try:
        with psycopg.connect(f"dbname={database}", connect_timeout=2):
            return True
    except psycopg.OperationalError:
        return False


def start_server(wait=60):
    """Install and start PostgreSQL if nothing is answering. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No PostgreSQL is answering. Start your own server and run this again: "
                           "this cell only installs one on Linux, which is what Colab runs.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"{sudo}apt-get -qq update")
    shell(f"{sudo}apt-get -qq -y install postgresql postgresql-contrib")
    shell(f"{sudo}service postgresql start")                        # Colab has no systemd

    for attempt in range(1, wait + 1):                              # start returns before it listens
        if shell("pg_isready -q")[0] == 0:
            break
        print(f"  waiting for the cluster ({attempt})")              # a silent minute looks hung
        time.sleep(1)
    else:
        raise RuntimeError(f"PostgreSQL did not accept connections within {wait} seconds.")

    me = getpass.getuser()                                          # peer authentication wants a role
    asking = f"""sudo -u postgres psql -tAc "SELECT 1 FROM pg_roles WHERE rolname='{me}'" """
    if shell(asking)[1] != "1":                                     # named for the operating system user
        shell(f"sudo -u postgres createuser -s {me}")
    return "installed and started"

def build(rows=5000):
    """Make the guide database and its events table, and fill it once."""
    with psycopg.connect("dbname=postgres", autocommit=True) as conn:
        if not conn.execute("SELECT 1 FROM pg_database WHERE datname = 'guide'").fetchone():
            conn.execute("CREATE DATABASE guide")                   # cannot run in a transaction

    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        for (leftover,) in conn.execute(                            # whatever an earlier run made
                "SELECT tablename FROM pg_tables "
                "WHERE schemaname = 'public' AND tablename <> 'events'").fetchall():
            conn.execute(f'DROP TABLE IF EXISTS "{leftover}" CASCADE')

        conn.execute("""CREATE TABLE IF NOT EXISTS events (
                            id bigserial PRIMARY KEY,
                            ts timestamptz NOT NULL DEFAULT now(),
                            kind text NOT NULL,
                            payload jsonb NOT NULL)""")
        if conn.execute("SELECT count(*) FROM events").fetchone()[0] == 0:
            conn.execute("""INSERT INTO events (kind, payload)
                            SELECT (ARRAY['click', 'view', 'purchase'])[1 + n %% 3],
                                   jsonb_build_object('n', n, 'size', 1 + n %% 7)
                            FROM generate_series(1, %s) AS n""", (rows,))
        return conn.execute("SELECT count(*) FROM events").fetchone()[0]

def report():
    """One line naming what this notebook is running against."""
    rows = build()                                                  # makes the database if it is new
    with psycopg.connect("dbname=guide") as conn:
        major = int(conn.execute("SHOW server_version_num").fetchone()[0]) // 10000
    return (f"PostgreSQL {major} | psycopg {version('psycopg')} | asyncpg {version('asyncpg')} "
            f"| events: {rows} rows")

class Money:
    """A class of your own, which the drivers have never heard of."""

    def __init__(self, amount):
        self.amount = Decimal(amount)

    def __repr__(self):
        return f"Money({self.amount})"


def build_prices():
    """One row with a column of each type this notebook is about."""
    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        conn.execute("DROP TABLE IF EXISTS prices")
        conn.execute("CREATE TABLE prices (id int, amount numeric(10,2), "
                     "at timestamptz, tags text[])")
        conn.execute("INSERT INTO prices VALUES "
                     "(1, 19.99, '2026-01-15 12:00:00+00', ARRAY['sea', 'stone'])")


def utc(**kwargs):
    """A connection whose session time zone is UTC, so a printed timestamp is the same everywhere."""
    return psycopg.connect("dbname=guide", options="-c TimeZone=UTC", **kwargs)


print("server:", start_server())
print(report())
build_prices()
print("prices is ready")


server: already running
PostgreSQL 16 | psycopg 3.3.6 | asyncpg 0.31.0 | events: 5000 rows
prices is ready


## Worked examples

### What already has a rule

Most of what you would pass already works, and `pg_typeof` says what the server decided it was:


In [2]:
values = [42, 3.5, "text", True, None, Decimal("1.25"),
          datetime.date(2026, 1, 15), [1, 2, 3], {"a": 1}]

with psycopg.connect("dbname=guide") as conn:
    for value in values:
        try:
            back = conn.execute("SELECT %s", (value,)).fetchone()[0]
            print(f"  {type(value).__name__:<9} {str(value)[:16]:<18} -> back as "
                  f"{type(back).__name__:<9} {back!r}")
        except psycopg.ProgrammingError as error:
            conn.rollback()
            print(f"  {type(value).__name__:<9} {str(value)[:16]:<18} -> {error}")


  int       42                 -> back as int       42
  float     3.5                -> back as float     3.5
  str       text               -> back as str       'text'
  bool      True               -> back as bool      True
  NoneType  None               -> back as NoneType  None
  Decimal   1.25               -> back as Decimal   Decimal('1.25')
  date      2026-01-15         -> back as date      datetime.date(2026, 1, 15)
  list      [1, 2, 3]          -> back as list      [1, 2, 3]
  dict      {'a': 1}           -> cannot adapt type 'dict' using placeholder '%s' (format: AUTO)


Everything on that list made the round trip except the `dict`, which is the surprise and the subject
of **JSONB**: psycopg will not assume a dictionary means JSON, because it could as easily mean
`hstore` or a composite type.

Two of them came back as a different class from the one that went out. A `float` is still a `float`,
but the `Decimal` stayed a `Decimal` where a lesser type system would have handed back a float, and
that distinction is the next section.

### numeric comes back as Decimal

Exactness is the reason, and the consequence is a `TypeError` in code that has nothing to do with
the database:


In [3]:
with psycopg.connect("dbname=guide") as conn:
    amount = conn.execute("SELECT amount FROM prices").fetchone()[0]

print("type:", type(amount).__name__, "| value:", amount)

try:
    print(amount * 1.2)
except TypeError as error:
    print("times a float:  ", type(error).__name__ + ":", error)

print("times a Decimal:", amount * Decimal("1.2"))
print("as a float:     ", float(amount) * 1.2)


type: Decimal | value: 19.99
times a float:   TypeError: unsupported operand type(s) for *: 'decimal.Decimal' and 'float'
times a Decimal: 23.988
as a float:      23.987999999999996


`float(amount) * 1.2` is the line to be careful with. It works, and it is the wrong answer for money,
because that is exactly the rounding `numeric` exists to avoid. Convert at the edge where a float is
genuinely wanted, and keep `Decimal` everywhere else.

### A timestamp moves with the session

`timestamptz` stores an instant. What you see is that instant rendered in the session's time zone,
and the session's time zone is a setting:


In [4]:
for zone in ("UTC", "America/New_York", "Asia/Tokyo"):
    with psycopg.connect("dbname=guide", options=f"-c TimeZone={zone}") as conn:
        moment = conn.execute("SELECT at FROM prices").fetchone()[0]
    print(f"  {zone:<18} {moment.isoformat()}")


  UTC                2026-01-15T12:00:00+00:00
  America/New_York   2026-01-15T07:00:00-05:00
  Asia/Tokyo         2026-01-15T21:00:00+09:00


Three different strings, one instant: all three are the same moment, and `==` between any two of
them is `True`. The trap is not that the value changed, because it did not. It is that a report
printing local times is printing them in whatever zone the connection happened to have, which on a
server is usually not the one anybody meant.

Say which zone you want, per connection, and the question does not come up:


In [5]:
with utc() as conn:
    a = conn.execute("SELECT at FROM prices").fetchone()[0]
with psycopg.connect("dbname=guide", options="-c TimeZone=Asia/Tokyo") as conn:
    b = conn.execute("SELECT at FROM prices").fetchone()[0]

print("printed differently:", a.isoformat(), "and", b.isoformat())
print("the same instant:   ", a == b)


printed differently: 2026-01-15T12:00:00+00:00 and 2026-01-15T21:00:00+09:00
the same instant:    True


### An integer that stops being an integer

PostgreSQL's `bigint` holds up to about nine quintillion. A Python `int` does not stop there:


In [6]:
with psycopg.connect("dbname=guide") as conn:
    for value in (42, 2**40, 2**70):
        named = conn.execute("SELECT pg_typeof(%s)::text", (value,)).fetchone()[0]
        came_back = conn.execute("SELECT %s", (value,)).fetchone()[0]
        print(f"  2**{value.bit_length() - 1:<3} {named:<9} back as {type(came_back).__name__}")


  2**5   smallint  back as int
  2**40  bigint    back as int
  2**70  numeric   back as Decimal


Past `bigint` the value is sent as `numeric`, and `numeric` loads as `Decimal`, so an integer goes
out and a `Decimal` comes back. That is an adaptation fact and nothing more: what a query does with a
`numeric` where it expected a `bigint` is the query's business, and this notebook makes no claim
about it.

### A rule of your own, both ways

The first look wrote a dumper. Here is the pair, so a `Money` goes out and a `Money` comes back:


In [7]:
class MoneyDumper(Dumper):
    oid = psycopg.adapters.types["numeric"].oid

    def dump(self, obj):
        return str(obj.amount).encode()


class MoneyLoader(Loader):
    def load(self, data):
        return Money(bytes(data).decode())


with psycopg.connect("dbname=guide") as conn:
    conn.adapters.register_dumper(Money, MoneyDumper)
    conn.adapters.register_loader("numeric", MoneyLoader)

    conn.execute("INSERT INTO prices VALUES (2, %s, now(), ARRAY['sea'])", (Money("4.50"),))
    got = conn.execute("SELECT amount FROM prices ORDER BY id").fetchall()
    print("both ways:", got)
    conn.rollback()


both ways: [(Money(19.99),), (Money(4.50),)]


`register_loader("numeric", ...)` is doing something worth noticing: it changes what **every**
`numeric` on that connection loads as, not just the ones that came from a `Money`. PostgreSQL sends
a type, not a class, so a loader is chosen by the column's type and cannot know what you meant.

That is why both registrations above are on the connection rather than on `psycopg.adapters`, which
would change it for the whole process.

### Arrays

A PostgreSQL array is a type, and a Python list is its rule. That is what made `= ANY(%s)` work in
**Placeholders and Identifiers**:


In [8]:
with psycopg.connect("dbname=guide") as conn:
    tags = conn.execute("SELECT tags FROM prices WHERE id = 1").fetchone()[0]
    print("a text[] comes back as:", type(tags).__name__, tags)

    print("a list goes out as:   ",
          conn.execute("SELECT pg_typeof(%s)::text", ([1, 2, 3],)).fetchone()[0])
    print("and matches with ANY: ",
          conn.execute("SELECT count(*) FROM events WHERE id = ANY(%s)", ([1, 2, 3],)).fetchone()[0])
    print("nested lists survive: ",
          conn.execute("SELECT %s", ([[1, 2], [3, 4]],)).fetchone()[0])


a text[] comes back as: list ['sea', 'stone']
a list goes out as:    smallint[]
and matches with ANY:  3
nested lists survive:  [[1, 2], [3, 4]]


An empty list is the one case with no answer, because there is nothing in it to take a type from, and
psycopg needs a cast to be told: `%s::int[]`.

### asyncpg, and the JSONB default

asyncpg ships with fewer rules on purpose. The one everybody meets is JSONB:


In [9]:
conn = await asyncpg.connect(database="guide")

raw = await conn.fetchval("SELECT payload FROM events ORDER BY id LIMIT 1")
print("a jsonb column arrives as:", type(raw).__name__, repr(raw))

try:
    await conn.fetchval("SELECT $1::jsonb", {"a": 1})
except asyncpg.exceptions.DataError as error:
    print("a dict as a parameter:   ", type(error).__name__ + ":", error)


a jsonb column arrives as: str '{"n": 1, "size": 2}'
a dict as a parameter:    DataError: invalid input for query argument $1: {'a': 1} (expected str, got dict)


It came back as the text it is stored as, and a dictionary going the other way is refused. One call
fixes both directions at once, which is asyncpg's whole adaptation interface:


In [10]:
await conn.set_type_codec("jsonb", encoder=json.dumps, decoder=json.loads, schema="pg_catalog")

after = await conn.fetchval("SELECT payload FROM events ORDER BY id LIMIT 1")
print("now it arrives as:", type(after).__name__, after)
print("and a dict goes over:", await conn.fetchval("SELECT $1::jsonb", {"a": 1}))
await conn.close()


now it arrives as: dict {'n': 1, 'size': 2}
and a dict goes over: {'a': 1}


`encoder` is psycopg's dumper and `decoder` is its loader, registered together and scoped to the
connection the same way. The `schema="pg_catalog"` is there because `jsonb` is a built-in type;
a type you created yourself lives in `public` or wherever you made it.

### When to reach for which

| What you have | What to do |
|---|---|
| an ordinary Python type | nothing, there is a rule already |
| a `dict` you mean as JSON | wrap it, which is **JSONB** |
| a class of your own, going in | a `Dumper` with an `oid`, registered |
| a class of your own, coming back | a `Loader` registered against the column's type |
| both, in asyncpg | one `set_type_codec` with an encoder and a decoder |
| money | leave it as `Decimal`, and convert only at the edge |
| a timestamp anybody will read | set the session `TimeZone` on the connection |
| a list to match against | pass the list, and cast an empty one |

Register on the connection unless you mean it for the whole process. A loader is chosen by the
PostgreSQL type, so registering one for `numeric` changes every `numeric`, which is rarely what you
want outside a program that owns its schema.

### A price list that survives the round trip, finished

Everything above, as the thing it is for: money that stays exact, timestamps that print the same
wherever this runs, and a class that goes in and comes back as itself.


In [11]:
def price_list(rows):
    """Write prices as Money, read them back as Money, and print the times in UTC."""
    with utc() as conn:
        conn.adapters.register_dumper(Money, MoneyDumper)
        conn.adapters.register_loader("numeric", MoneyLoader)

        conn.execute("TRUNCATE prices")
        for identifier, amount, tags in rows:
            conn.execute("INSERT INTO prices VALUES (%s, %s, %s, %s)",
                         (identifier, Money(amount), datetime.datetime(2026, 1, 15, 12, 0,
                                                                       tzinfo=datetime.timezone.utc),
                          tags))

        return conn.execute("SELECT id, amount, at, tags FROM prices ORDER BY id").fetchall()


for identifier, amount, at, tags in price_list([(1, "19.99", ["sea"]), (2, "4.50", ["stone", "salt"])]):
    print(f"  {identifier}  {amount!r:<16} {at.isoformat()}  {tags}")

build_prices()


  1  Money(19.99)     2026-01-15T12:00:00+00:00  ['sea']
  2  Money(4.50)      2026-01-15T12:00:00+00:00  ['stone', 'salt']


The amounts are `Money` in both directions, the times are the same string on any machine because the
connection said so, and the tags went over as a list without anything being written about arrays.

### Where each part came from

| In the price list | What it relies on | The section that showed it |
|---|---|---|
| `register_dumper(Money, ...)` | a rule for the way in | A rule of your own |
| `register_loader("numeric", ...)` | a rule for the way back, by column type | A rule of your own |
| `utc()` | a session time zone that is not the machine's | A timestamp moves |
| `Money` holding a `Decimal` | exactness kept away from floats | numeric comes back as Decimal |
| the tags passed as a list | the array rule that is already there | Arrays |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/asyncpg-and-psycopg3-deep-dive/05-types-and-adaptation-solutions.ipynb).

**1.** Print what the server calls each of five different Python values passed as parameters.


In [12]:
# your code here


**2.** Read the `amount` column, show its type, and show what happens when it meets a float and what
happens when it meets a `Decimal`.


In [13]:
# your code here


**3.** Read the same `timestamptz` through two connections with different session time zones, and
show the two values are equal.


In [14]:
# your code here


**4.** Write a dumper for a class of your own and use it as a parameter.


In [15]:
# your code here


**5.** Pass a Python list as a parameter and show what type the server made of it.


In [16]:
# your code here


**6.** Read a JSONB column through asyncpg before and after registering a codec for it.


In [17]:
# your code here


## Common errors

### psycopg.ProgrammingError: cannot adapt type 'Money' using placeholder '%s' (format: AUTO)


In [18]:
with psycopg.connect("dbname=guide") as conn:
    conn.execute("SELECT %s", (Money("19.99"),))


ProgrammingError: cannot adapt type 'Money' using placeholder '%s' (format: AUTO)

There is no rule for this class, and psycopg will not invent one. The message names the class and the
placeholder, and `format: AUTO` is psycopg saying it tried both the text and the binary form.

Two answers, and which is right depends on whether the class is yours to teach:


In [19]:
with psycopg.connect("dbname=guide") as conn:
    print("unwrap it at the call:", conn.execute("SELECT %s", (Money("19.99").amount,)).fetchone())

    conn.adapters.register_dumper(Money, MoneyDumper)
    print("or register a rule:   ", conn.execute("SELECT %s", (Money("19.99"),)).fetchone())


unwrap it at the call: (Decimal('19.99'),)
or register a rule:    (Decimal('19.99'),)


### TypeError: unsupported operand type(s) for *: 'decimal.Decimal' and 'float'


In [20]:
with psycopg.connect("dbname=guide") as conn:
    amount = conn.execute("SELECT amount FROM prices WHERE id = 1").fetchone()[0]

amount * 1.075


TypeError: unsupported operand type(s) for *: 'decimal.Decimal' and 'float'

Not a database error, and it usually happens a long way from the query, which is what makes it
confusing: a tax rate written as a float meets a column that loaded as `Decimal`.

Python is refusing on purpose. Mixing them would quietly turn an exact number into an approximate
one, which for money is the bug the `numeric` column was chosen to prevent:


In [21]:
print("as Decimal: ", amount * Decimal("1.075"))
print("as float:   ", float(amount) * 1.075, "<- fine for a ratio, wrong for money")
print("0.1 + 0.2 as floats:  ", 0.1 + 0.2)
print("0.1 + 0.2 as Decimals:", Decimal("0.1") + Decimal("0.2"))


as Decimal:  21.48925
as float:    21.48925 <- fine for a ratio, wrong for money
0.1 + 0.2 as floats:   0.30000000000000004
0.1 + 0.2 as Decimals: 0.3


### asyncpg.exceptions.DataError: invalid input for query argument $1


In [22]:
conn = await asyncpg.connect(database="guide")
try:
    await conn.fetchval("SELECT $1::jsonb", {"a": 1})
except asyncpg.exceptions.DataError as error:
    print(type(error).__name__ + ":", error)
finally:
    await conn.close()


DataError: invalid input for query argument $1: {'a': 1} (expected str, got dict)


asyncpg does no conversion it has not been told about, and it says what it expected and what it got.
psycopg's message for the same situation names the class instead, which is the same refusal worded
from the other end.

`set_type_codec` is the fix, and it is worth registering once where the connection is made rather
than at the call site:


In [23]:
async def connect_with_json():
    """A connection that speaks JSON in both directions."""
    conn = await asyncpg.connect(database="guide")
    await conn.set_type_codec("jsonb", encoder=json.dumps, decoder=json.loads, schema="pg_catalog")
    return conn


conn = await connect_with_json()
print("a dict goes over:  ", await conn.fetchval("SELECT $1::jsonb", {"a": 1}))
print("and comes back as: ", type(await conn.fetchval("SELECT payload FROM events LIMIT 1")).__name__)
await conn.close()


a dict goes over:   {'a': 1}
and comes back as:  dict


### No error, and a timestamp nobody can agree on: a session time zone nobody set


In [24]:
readings = []
for zone in ("UTC", "America/New_York"):
    with psycopg.connect("dbname=guide", options=f"-c TimeZone={zone}") as conn:
        readings.append(conn.execute("SELECT at FROM prices WHERE id = 1").fetchone()[0])

print("one connection says:", readings[0].isoformat())
print("the other says:    ", readings[1].isoformat())
print("they are equal:    ", readings[0] == readings[1])
print("the hour differs:  ", readings[0].hour, "against", readings[1].hour)


one connection says: 2026-01-15T12:00:00+00:00
the other says:     2026-01-15T07:00:00-05:00
they are equal:     True
the hour differs:   12 against 7


Both are right, which is the problem. The instant is the same and `==` agrees, but anything that
formats an hour, groups by a day or compares against a date literal will give a different answer
depending on which connection it ran on.

The default comes from the server's configuration, so it is whatever the machine was set up with and
is rarely the same in development and production. Set it where connections are made:


In [25]:
with utc() as conn:
    print("this notebook's connections use:",
          conn.execute("SHOW TimeZone").fetchone()[0])
    print("and group by a day the same way everywhere:",
          conn.execute("SELECT date_trunc('day', at) FROM prices WHERE id = 1").fetchone()[0].isoformat())


this notebook's connections use: UTC
and group by a day the same way everywhere: 2026-01-15T00:00:00+00:00


## Recap

- Every value crossing the socket becomes bytes. A **dumper** writes them on the way in and a
  **loader** reads them on the way back.
- Ordinary Python types already have rules. A class of your own does not, and psycopg refuses rather
  than guessing, which is the `cannot adapt type` message.
- A dumper needs an `oid` saying which PostgreSQL type it is producing. A loader is registered
  against a PostgreSQL type, so it changes every column of that type on the connection.
- `numeric` loads as `Decimal`, which will not mix with `float`. Converting is easy and is the wrong
  thing to do to money.
- `timestamptz` is one instant rendered in the session's time zone. Set `TimeZone` on the connection
  or your output depends on the machine.
- An integer larger than `bigint` is sent as `numeric` and comes back as `Decimal`.
- A Python list is a PostgreSQL array, which is what makes `= ANY(%s)` work. An empty one needs a
  cast.
- asyncpg registers both directions at once with `set_type_codec`, and ships without a JSONB codec,
  so JSONB arrives as a string until you add one.


## What is next

The **JSONB** notebook is the type this one kept pointing at: why a dictionary has to be wrapped
rather than guessed, the containment operator that makes JSONB worth using, and what `EXPLAIN` says
about the same query before and after a GIN index.


---

&#8592; **Previous:** [Placeholders and Identifiers](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/asyncpg-and-psycopg3-deep-dive/04-placeholders-and-identifiers.ipynb)  &nbsp;·&nbsp;  [asyncpg and psycopg3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)
